# 02 - Fidelity kernel (week 1, first version)
Owner: **Farrukh**

Fidelity quantum kernel $k(x,x') = |\langle\psi(x)|\psi(x')\rangle|^2$ on Fashion-MNIST (T-shirt/top vs. Shirt).

- Main numbers: **sampled** kernel on AerSimulator (NB11 protocol).
- Reference: **exact** kernel (no shot noise).
- Baseline: **1-layer** kernel, which has a classical closed form.

$\alpha$ and $C$ are tuned by inner cross-validation on training data only. The test set is used once, at the end.

In [1]:
import sys, json, time
sys.path.insert(0, "..")
import numpy as np
import qiskit
from src.common import (get_split, make_preprocessor, encoding_circuit, inner_folds,
                        cv_score_precomputed, evaluate_precomputed, mean_offdiag,
                        min_eig, result_path, SHOTS)
from src.fidelity import (fidelity_kernel_sampled, fidelity_kernel_exact,
                          product_kernel_closed, overlap_resources)
print("qiskit", qiskit.__version__)

qiskit 2.6.2


## 1. Settings

In [2]:
SEED     = 0                       # first seed; all 5 seeds come in week 2
N_QUBITS = 4                       # = number of PCA features
LAYERS   = 2                       # main setting (1 = classically easy baseline)
ALPHAS   = [0.5, 1.0, 2.0, np.pi]  # encoding scale, tuned on inner CV
CS       = [0.001, 0.01, 0.1, 1, 10, 100, 1000]  # SVM C, same grid as classical baselines

## 2. Data (downloads Fashion-MNIST once, ~30 MB)

In [3]:
Xtr, ytr, Xte, yte = get_split(SEED)
print("train", Xtr.shape, "test", Xte.shape)
print("train balance", np.bincount((ytr + 1) // 2), "| test balance", np.bincount((yte + 1) // 2))
print("majority-class rate: %.3f" % (max(np.bincount((yte + 1) // 2)) / len(yte)))

train (80, 784) test (40, 784)
train balance [40 40] | test balance [20 20]
majority-class rate: 0.500


## 3. Tune alpha and C on inner CV (training data only, sampled kernel)

In [4]:
enc = encoding_circuit(N_QUBITS, LAYERS)
folds = inner_folds(ytr, SEED)
best = dict(score=-1)
t0 = time.time()
print(f"{'alpha':>7}{'C':>6}{'inner acc':>11}{'mean K':>9}")
for a in ALPHAS:
    tf = make_preprocessor(N_QUBITS, a)(Xtr)
    K = fidelity_kernel_sampled(enc, tf(Xtr), seed=SEED)
    for C in CS:
        s = cv_score_precomputed(K, ytr, folds, C)
        print(f"{a:>7.3f}{C:>6}{s:>11.3f}{mean_offdiag(K):>9.3f}")
        if s > best["score"]:
            best = dict(score=s, alpha=a, C=C)
print(f"best: alpha={best['alpha']:.3f}, C={best['C']}  ({time.time()-t0:.0f} s)")

  alpha     C  inner acc   mean K
  0.500 0.001      0.775    0.819
  0.500  0.01      0.775    0.819
  0.500   0.1      0.775    0.819
  0.500     1      0.762    0.819
  0.500    10      0.775    0.819
  0.500   100      0.775    0.819
  0.500  1000      0.762    0.819
  1.000 0.001      0.800    0.510
  1.000  0.01      0.800    0.510
  1.000   0.1      0.800    0.510
  1.000     1      0.762    0.510
  1.000    10      0.825    0.510
  1.000   100      0.762    0.510
  1.000  1000      0.738    0.510
  2.000 0.001      0.787    0.240
  2.000  0.01      0.787    0.240
  2.000   0.1      0.787    0.240
  2.000     1      0.762    0.240
  2.000    10      0.775    0.240
  2.000   100      0.700    0.240
  2.000  1000      0.713    0.240
  3.142 0.001      0.762    0.175
  3.142  0.01      0.762    0.175
  3.142   0.1      0.762    0.175
  3.142     1      0.738    0.175
  3.142    10      0.700    0.175
  3.142   100      0.688    0.175
  3.142  1000      0.662    0.175
best: alpha=1.

## 4. Test once with the chosen alpha and C

In [5]:
tf = make_preprocessor(N_QUBITS, best["alpha"])(Xtr)
Atr, Ate = tf(Xtr), tf(Xte)

K_s,  Kt_s = fidelity_kernel_sampled(enc, Atr, seed=SEED), fidelity_kernel_sampled(enc, Ate, Atr, seed=SEED)
K_e,  Kt_e = fidelity_kernel_exact(enc, Atr), fidelity_kernel_exact(enc, Ate, Atr)

# 1-layer baseline: closed form, alpha and C tuned separately (cheap, no circuits)
best1 = dict(score=-1)
for a in ALPHAS:
    tf1 = make_preprocessor(N_QUBITS, a)(Xtr)
    K1 = product_kernel_closed(tf1(Xtr))
    for C in CS:
        s = cv_score_precomputed(K1, ytr, folds, C)
        if s > best1["score"]:
            best1 = dict(score=s, alpha=a, C=C)
tf1 = make_preprocessor(N_QUBITS, best1["alpha"])(Xtr)
K_1, Kt_1 = product_kernel_closed(tf1(Xtr)), product_kernel_closed(tf1(Xte), tf1(Xtr))

rows = {
    f"Fidelity L={LAYERS} (sampled)": evaluate_precomputed(K_s, Kt_s, ytr, yte, best["C"]),
    f"Fidelity L={LAYERS} (exact ref)": evaluate_precomputed(K_e, Kt_e, ytr, yte, best["C"]),
    "Fidelity L=1 (closed form)": evaluate_precomputed(K_1, Kt_1, ytr, yte, best1["C"]),
}
print(f"{'model':<32}{'test acc':>9}{'test AUC':>9}")
for k, v in rows.items():
    print(f"{k:<32}{v['acc']:>9.3f}{v['auc']:>9.3f}")
print(f"\nsampled K: mean offdiag {mean_offdiag(K_s):.3f}, min eig {min_eig(K_s):+.1e}")
print(f"max |sampled - exact| on train kernel: {np.abs(K_s - K_e).max():.3f}")

model                            test acc test AUC
Fidelity L=2 (sampled)              0.800    0.885
Fidelity L=2 (exact ref)            0.825    0.935
Fidelity L=1 (closed form)          0.800    0.920

sampled K: mean offdiag 0.510, min eig -1.4e-01
max |sampled - exact| on train kernel: 0.035


## 5. Resources and save

In [6]:
res = overlap_resources(enc)
n_tr, n_te = len(ytr), len(yte)
circuits = n_tr * (n_tr - 1) // 2 + n_te * n_tr
res.update(circuits_per_eval=circuits, shots_per_eval=circuits * SHOTS)
print(res)

out = dict(seed=SEED, n_qubits=N_QUBITS, layers=LAYERS, shots=SHOTS,
           best=best, best_L1=best1, results=rows, resources=res,
           qiskit=qiskit.__version__)
path = result_path("tables", "fidelity_v1", SEED, N_QUBITS, "json")
path.write_text(json.dumps(out, indent=2))
np.save(result_path("kernels", "fidelity_sampled_train", SEED, N_QUBITS, "npy"), K_s)
np.save(result_path("kernels", "fidelity_exact_train", SEED, N_QUBITS, "npy"), K_e)
print("saved", path)

{'qubits': 4, 'depth': 14, 'two_qubit_gates': 12, 'circuits_per_eval': 6360, 'shots_per_eval': 13025280}
saved E:\LUMS\Semester 3\Quantum Machine Learning\Project\AI631-Projected-Quantum-Kernels\results\tables\fidelity_v1_seed0_q4.json
